# Obtención de datos de TMDB

En este notebook realizaré la extracción de los datos y guardaré
las respuestas originales para su posterior limpieza y análisis.

## 1. Importación de librerías y carga de credenciales

In [1]:
import calendar
import json
import os
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Localizamos el archivo .env desde la carpeta actual hacia arriba
ruta_env = find_dotenv(usecwd=True)

if not ruta_env:
    raise FileNotFoundError("No se ha encontrado el archivo .env.")

# Cargamos las variables de entorno
load_dotenv(ruta_env, override=True)
token = os.getenv("TMDB_TOKEN")

if not token or not token.strip():
    raise ValueError("Falta TMDB_TOKEN o está vacío en el archivo .env.")

print("Credenciales cargadas.")

Credenciales cargadas.


## 2. Configuración de las carpetas

Utilizo la ubicación del archivo `.env` para identificar la raíz del
proyecto y construir la ruta `src/data/raw`.

In [2]:
# La carpeta que contiene .env es la raíz de nuestro proyecto
raiz_proyecto = Path(ruta_env).resolve().parent

# Carpetas de datos originales, créditos y datos procesados
carpeta_raw = raiz_proyecto / "src" / "data" / "raw"
carpeta_creditos = carpeta_raw / "creditos"
carpeta_proc = raiz_proyecto / "src" / "data" / "processed"

for carpeta in (carpeta_raw, carpeta_creditos, carpeta_proc):
    carpeta.mkdir(parents=True, exist_ok=True)

print("Carpeta de datos originales:", carpeta_raw.relative_to(raiz_proyecto))

Carpeta de datos originales: src\data\raw


## 3. Configuración de la extracción

Defino el periodo 2000–2025 y los parámetros comunes de las consultas.
La descarga se organizará por meses, comprobando si algún intervalo
necesita subdividirse.

In [3]:
# Periodo de estudio: ambos años están incluidos
anio_inicio = 2000
anio_fin = 2025

# Dirección base de la API
url_base = "https://api.themoviedb.org/3"

# Cabecera de autorización que reutilizaremos en las solicitudes
headers = {
    "Authorization": f"Bearer {token}"
}

# Parámetros comunes de los listados de películas
parametros_comunes = {
    "language": "es-ES",
    "include_adult": "false"
}

print(f"Periodo de extracción: {anio_inicio}–{anio_fin}")

Periodo de extracción: 2000–2025


## 4. Preparación de los intervalos mensuales

Genero las fechas de inicio y fin de cada mes entre 2000 y 2025.
Estos intervalos cubren el periodo completo sin solaparse y tienen
en cuenta la duración de cada mes y los años bisiestos.

In [4]:
# Lista donde guardaremos los intervalos
intervalos = []

# Recorremos todos los años, incluido el último
for anio in range(anio_inicio, anio_fin + 1):

    # Para cada año, recorremos los doce meses
    for mes in range(1, 13):

        # Calculamos el último día del mes
        ultimo_dia = calendar.monthrange(anio, mes)[1]

        # Añadimos el intervalo mensual
        intervalos.append({
            "fecha_inicio": f"{anio}-{mes:02d}-01",
            "fecha_fin": f"{anio}-{mes:02d}-{ultimo_dia:02d}"
        })

# Creamos una tabla para revisar los intervalos
df_intervalos = pd.DataFrame(intervalos)

print("Número de intervalos:", len(df_intervalos))
display(df_intervalos.head())
display(df_intervalos.tail())

Número de intervalos: 312


,fecha_inicio,fecha_fin
0,2000-01-01,2000-01-31
1,2000-02-01,2000-02-29
2,2000-03-01,2000-03-31
3,2000-04-01,2000-04-30
4,2000-05-01,2000-05-31


,fecha_inicio,fecha_fin
307,2025-08-01,2025-08-31
308,2025-09-01,2025-09-30
309,2025-10-01,2025-10-31
310,2025-11-01,2025-11-30
311,2025-12-01,2025-12-31


Se generan 312 intervalos mensuales (26 años por 12 meses), desde enero
de 2000 hasta diciembre de 2025. El último día de cada mes se calcula
con `calendar.monthrange`, de modo que los años bisiestos quedan bien
recogidos (por ejemplo, febrero de 2000 termina el día 29).

## 5. Función de consulta del listado

Defino una función que recibe las fechas del intervalo y el número
de página. Comprueba los errores HTTP y devuelve los datos recibidos.

### 5.1. Lectura o descarga reanudable de una página

Antes de consultar la API, la función comprueba si la página ya está
guardada en `src/data/raw`. Si existe, la lee del disco; si no, la
descarga. Escribe primero en un archivo temporal y después lo renombra,
para que una interrupción no deje un JSON incompleto con el nombre
definitivo. Así la descarga se puede reanudar sin repetir lo hecho.

In [5]:
def consultar_pagina(fecha_inicio, fecha_fin, pagina=1):
    # Copiamos los parámetros comunes para conservar su configuración
    parametros = parametros_comunes.copy()

    # Añadimos las fechas y la página de esta consulta
    parametros.update({
        "primary_release_date.gte": fecha_inicio,
        "primary_release_date.lte": fecha_fin,
        "page": pagina
    })

    # Realizamos la solicitud
    respuesta = requests.get(
        f"{url_base}/discover/movie",
        headers=headers,
        params=parametros,
        timeout=30
    )

    # Detenemos la ejecución si hay un error HTTP
    respuesta.raise_for_status()

    # Devolvemos el contenido interpretado como un diccionario
    return respuesta.json()

In [6]:
def obtener_pagina(fecha_inicio, fecha_fin, pagina):
    # Construimos la ruta de esta página
    ruta_archivo = carpeta_raw / (
        f"Pelis_{fecha_inicio}_{fecha_fin}_pag_{pagina}.json"
    )

    if ruta_archivo.exists():
        # Recuperamos la respuesta guardada
        with open(ruta_archivo, "r", encoding="utf-8") as archivo:
            datos = json.load(archivo)
    else:
        # Consultamos la API si todavía no tenemos el archivo
        datos = consultar_pagina(
            fecha_inicio, fecha_fin, pagina=pagina
        )

        # Escribimos primero en un archivo temporal
        ruta_temporal = ruta_archivo.with_suffix(".tmp")

        with open(ruta_temporal, "w", encoding="utf-8") as archivo:
            json.dump(datos, archivo, ensure_ascii=False, indent=4)

        # El JSON definitivo aparece cuando termina la escritura
        ruta_temporal.replace(ruta_archivo)

    return datos

### 5.2. Descarga de todos los meses

Recorro los 312 intervalos. Para cada uno leo o descargo la primera
página, que indica cuántas páginas tiene el mes. Si algún intervalo
superara las 500 páginas, la ejecución se detiene con un error para
poder dividirlo. Después recorro las páginas restantes.

In [7]:
# Ponlo en True solo si quieres repetir la descarga de los listados
EJECUTAR_DESCARGA_PAGINAS = False

if EJECUTAR_DESCARGA_PAGINAS:
# Recorremos todos los meses del periodo configurado
for intervalo in intervalos:
    fecha_inicio = intervalo["fecha_inicio"]
    fecha_fin = intervalo["fecha_fin"]

    # Leemos o descargamos la primera página
    datos = obtener_pagina(fecha_inicio, fecha_fin, pagina=1)
    total_paginas = datos["total_pages"]

    # Detenemos la descarga si necesitamos dividir el intervalo
    if total_paginas > 500:
        raise ValueError(
            f"El intervalo {fecha_inicio} a {fecha_fin} tiene "
            f"{total_paginas} páginas y necesita dividirse."
        )

    # Leemos o descargamos las páginas restantes
    for pagina in range(2, total_paginas + 1):
        obtener_pagina(fecha_inicio, fecha_fin, pagina=pagina)

    print(
        f"{fecha_inicio} a {fecha_fin}: "
        f"{total_paginas} páginas disponibles en disco."
    )

IndentationError: expected an indented block after 'if' statement on line 4 (4602119.py, line 6)

Este bucle se ejecutó una vez. La celda incluye un interruptor para no
repetir la descarga por accidente.

## 6. Obtención de los créditos de las películas

Consulto los créditos con el identificador de cada película para
obtener las personas responsables de su dirección.

Primero pruebo con una película para revisar la estructura de la
respuesta. Después preparo el guardado, la descarga reanudable y la
descarga completa.

### 6.1. Lectura de las películas preparadas

Cargo la tabla de películas que se creó y limpió en el notebook
`02_limpieza_datos`, y que guardé como `peliculas_preparadas.pkl`.
Por eso este apartado se ejecutó después de la limpieza, y no justo
después de la descarga de los listados.

La necesito para obtener los identificadores de las películas y
consultar sus créditos.

In [8]:
# Tabla creada y guardada al final del notebook 02_limpieza_datos
df_peliculas = pd.read_pickle(carpeta_proc / "peliculas_preparadas.pkl")

# Comprobamos que se ha cargado bien con las primeras filas
display(df_peliculas[["id", "title"]].head())

,id,title
0,271517,Apartment Hunting
1,49948,Fantasía 2000
2,122163,Jill Rips
3,453614,Codicia mortal
4,841159,Rata callejera


### 6.2. Prueba de consulta de créditos

Consulto los créditos de la primera película del DataFrame utilizando
su identificador. Reviso las claves de la respuesta y las primeras
personas del equipo técnico (`crew`).

In [9]:
# Seleccionamos el identificador de la primera película
id_pelicula = int(df_peliculas["id"].iloc[0])

# Solicitamos sus créditos
respuesta = requests.get(
    f"{url_base}/movie/{id_pelicula}/credits",
    headers=headers,
    params={"language": "es-ES"},
    timeout=30
)

# Comprobamos si la solicitud ha fallado
respuesta.raise_for_status()

# Convertimos la respuesta JSON en un diccionario
creditos = respuesta.json()

print("Película:", df_peliculas["title"].iloc[0])
print("Claves de la respuesta:", creditos.keys())

display(pd.DataFrame(creditos["crew"]).head())

Película: Apartment Hunting
Claves de la respuesta: dict_keys(['id', 'cast', 'crew'])


,adult,gender,id,known_for_department,name,original_name,popularity,profile_path,credit_id,department,job
0,False,1,2304648,Directing,Annie Bradley,Annie Bradley,0.3054,/ayBbe24fyWtT1WRUJEI7mHqpAKM.jpg,645faa8ee3fa2f0103a0baca,Directing,Second Unit Director
1,False,1,2304648,Directing,Annie Bradley,Annie Bradley,0.3054,/ayBbe24fyWtT1WRUJEI7mHqpAKM.jpg,645faa88a67254014366578b,Directing,First Assistant Director
2,False,0,961445,Crew,Brock Jolliffe,Brock Jolliffe,0.4455,NaN,647ea3d00e29a22be293a5c1,Crew,Special Effects
3,False,0,2547984,Production,Michael Coutanche,Michael Coutanche,0.1251,NaN,5e52eb76a76ac50013a57ce3,Production,Producer
4,False,0,1400524,Directing,Bill Robertson,Bill Robertson,0.2026,NaN,5e52eb3df48b340015719e9e,Directing,Director


### 6.3. Identificación de la dirección

Selecciono las personas cuyo puesto en los créditos es `Director`.
Conservo su identificador, nombre y código de género informado por TMDB.
Una película puede tener varias personas responsables de su dirección.

In [10]:
# Convertimos el equipo técnico en una tabla
df_crew_prueba = pd.DataFrame(creditos["crew"])

# Seleccionamos únicamente el puesto de Director
df_direccion_prueba = df_crew_prueba.loc[
    df_crew_prueba["job"] == "Director",
    ["id", "name", "gender", "job"]
].copy()

display(df_direccion_prueba)

,id,name,gender,job
4,1400524,Bill Robertson,0,Director


### 6.4. Guardado de los créditos de prueba

Guardo la respuesta completa de créditos en un archivo JSON identificado
con el ID de la película. Así podré reutilizarla sin repetir la consulta.

El género sin especificar se conserva como información desconocida.

In [11]:
# Guardamos los créditos de la película consultada
ruta_creditos = carpeta_creditos / f"creditos_{id_pelicula}.json"

with open(ruta_creditos, "w", encoding="utf-8") as f:
    json.dump(creditos, f, ensure_ascii=False, indent=2)

print("Créditos guardados en:", ruta_creditos.relative_to(raiz_proyecto))

Créditos guardados en: src\data\raw\creditos\creditos_271517.json


### 6.5. Reintentos ante errores temporales

Configuro una política de reintentos para las consultas ante errores
temporales de conexión o del servidor. Si la API responde con un límite
de solicitudes (429), se respeta el tiempo de espera indicado en la
cabecera `Retry-After`.

In [12]:
reintentos = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
)

### 6.6. Una sesión por hilo

Para consultar varias películas a la vez, cada hilo dispone de su propia
sesión con la política de reintentos anterior. Las sesiones de `requests`
no están garantizadas como seguras entre hilos, y reutilizar la conexión
de cada hilo evita abrir una conexión nueva en cada consulta.

In [13]:
# Guardamos una sesión independiente para cada hilo
sesiones_hilos = threading.local()

def obtener_sesion():
    # Creamos la sesión solo la primera vez que este hilo la necesita
    if not hasattr(sesiones_hilos, "sesion"):
        sesion_hilo = requests.Session()
        sesion_hilo.mount("https://", HTTPAdapter(max_retries=reintentos))
        sesiones_hilos.sesion = sesion_hilo

    return sesiones_hilos.sesion

### 6.7. Consulta y guardado reanudable de créditos

La función comprueba si los créditos de una película ya están guardados.
Si existen, los lee; si faltan, los consulta con la sesión de reintentos
del apartado anterior y los guarda en JSON.

Primero escribe un archivo temporal y después lo renombra, para evitar
que una interrupción deje un JSON incompleto con el nombre definitivo.

In [14]:
def obtener_creditos(id_pelicula):
    id_pelicula = int(id_pelicula)
    ruta = carpeta_creditos / f"creditos_{id_pelicula}.json"

    # Leemos los créditos si ya están guardados
    if ruta.exists():
        with open(ruta, "r", encoding="utf-8") as f:
            return json.load(f)

    # Descargamos los créditos que faltan
    respuesta = obtener_sesion().get(
        f"{url_base}/movie/{id_pelicula}/credits",
        headers=headers,
        params={"language": "es-ES"},
        timeout=30
    )
    respuesta.raise_for_status()
    datos = respuesta.json()

    # Comprobamos la estructura antes de guardar
    if datos.get("id") != id_pelicula or not isinstance(datos.get("crew"), list):
        raise ValueError(f"Créditos no válidos para la película {id_pelicula}")

    # Guardamos primero en un archivo temporal
    ruta_temporal = ruta.with_suffix(".tmp")

    with open(ruta_temporal, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False)

    ruta_temporal.replace(ruta)

    return datos

In [15]:
creditos_prueba = obtener_creditos(id_pelicula)
print("ID de la película:", creditos_prueba["id"])

ID de la película: 271517


### 6.8. Prueba con diez películas

Recorro los identificadores de las diez primeras películas y obtengo
sus créditos. Los archivos existentes se reutilizan y los que faltan
se descargan y guardan.

Mido el tiempo del lote como primera referencia del coste de descarga.

In [16]:
inicio = time.perf_counter()

# Seleccionamos diez películas para la prueba
ids_prueba = df_peliculas["id"].iloc[10:20]

for numero, id_actual in enumerate(ids_prueba, start=1):
    obtener_creditos(id_actual)
    print(f"{numero}/10: créditos disponibles para {id_actual}")

    # Dejamos una pequeña pausa entre consultas
    time.sleep(0.2)

segundos = time.perf_counter() - inicio
print(f"Tiempo del lote: {segundos:.1f} segundos")

1/10: créditos disponibles para 815679
2/10: créditos disponibles para 1421558
3/10: créditos disponibles para 108779
4/10: créditos disponibles para 7275
5/10: créditos disponibles para 1606212
6/10: créditos disponibles para 1267499
7/10: créditos disponibles para 1209381
8/10: créditos disponibles para 557429
9/10: créditos disponibles para 677737
10/10: créditos disponibles para 605594
Tiempo del lote: 2.2 segundos


En esta ejecución, la prueba con diez películas finaliza sin errores en
unos 2 segundos. Los créditos ya estaban guardados, por lo que ese
tiempo corresponde sobre todo a lectura local y a las pausas de 0,2
segundos entre consultas, y no a la velocidad real de la API.

En la primera ejecución, con descargas reales, el lote tardó 6,5
segundos. Extrapolado a las 681.868 películas, una descarga secuencial
completa habría durado unos cinco días. Por eso se probó la descarga
con varios hilos.

### 6.9. Prueba concurrente con cien películas

Utilizo cuatro trabajadores para consultar varias películas a la vez.
Cada trabajador dispone de su propia sesión con reintentos.

Pruebo con cien películas y registro los errores para poder
revisarlos. Los créditos descargados se guardan individualmente.

In [17]:
ids_prueba = df_peliculas["id"].iloc[20:120].tolist()
errores_prueba = []
correctos = 0
inicio = time.perf_counter()

with ThreadPoolExecutor(max_workers=4) as ejecutor:
    tareas = {
        ejecutor.submit(obtener_creditos, id_actual): id_actual
        for id_actual in ids_prueba
    }

    for tarea in as_completed(tareas):
        id_actual = tareas[tarea]

        try:
            tarea.result()
            correctos += 1
        except (requests.RequestException, ValueError) as error:
            errores_prueba.append({
                "id": id_actual,
                "error": str(error)
            })

segundos = time.perf_counter() - inicio

print("Créditos disponibles:", correctos)
print("Errores:", len(errores_prueba))
print(f"Tiempo del lote: {segundos:.1f} segundos")

if errores_prueba:
    display(pd.DataFrame(errores_prueba))

Créditos disponibles: 100
Errores: 0
Tiempo del lote: 0.2 segundos


La prueba con cien películas y cuatro trabajadores finaliza
correctamente:

- Créditos disponibles: 100.
- Errores registrados: 0.
- Tiempo de ejecución: 0,2 segundos.

Como los archivos ya estaban en disco, el tiempo observado corresponde
a lectura local y no debe interpretarse como velocidad de descarga de
la API.

### 6.10. Descarga completa de los créditos

Recorro todas las películas en lotes de 500, con cuatro trabajadores.
Los créditos existentes se reutilizan y los que faltan se descargan.

Registro los errores en un archivo JSONL, con una incidencia por línea.
Los errores de autorización detienen la ejecución y los errores de
escritura se propagan para evitar continuar si hay un problema de disco.

Al terminar cada lote muestro el progreso y el tiempo transcurrido.
La ejecución puede reanudarse conservando los JSON ya guardados.

La descarga completa se realizó una vez y tardó 487,8 minutos.

In [ ]:


ids_peliculas = df_peliculas["id"].drop_duplicates().tolist()
tamano_lote = 500
total = len(ids_peliculas)

ruta_errores = carpeta_creditos / "errores_creditos.jsonl"

correctos = 0
errores = 0
inicio = time.perf_counter()

with ThreadPoolExecutor(max_workers=4) as ejecutor:
    for posicion in range(0, total, tamano_lote):
        lote = ids_peliculas[posicion:posicion + tamano_lote]

        # Solo mantenemos las tareas de este lote
        tareas = {
            ejecutor.submit(obtener_creditos, id_actual): id_actual
            for id_actual in lote
        }

        for tarea in as_completed(tareas):
            id_actual = tareas[tarea]

            try:
                tarea.result()
                correctos += 1

            except (requests.RequestException, ValueError) as error:
                # Detenemos la ejecución si falla la autorización
                respuesta_error = getattr(error, "response", None)

                if (
                    respuesta_error is not None
                    and respuesta_error.status_code in (401, 403)
                ):
                    raise

                errores += 1

                # Añadimos el error al registro sin borrar los anteriores
                incidencia = {
                    "fecha": datetime.now(timezone.utc).isoformat(),
                    "id": int(id_actual),
                    "error": str(error)
                }

                with open(ruta_errores, "a", encoding="utf-8") as f:
                    f.write(json.dumps(incidencia, ensure_ascii=False) + "\n")

        revisados = correctos + errores
        minutos = (time.perf_counter() - inicio) / 60

        print(
            f"Revisadas: {revisados}/{total} "
            f"({revisados / total:.1%}) | "
            f"Disponibles: {correctos} | Errores: {errores} | "
            f"Tiempo: {minutos:.1f} min",
            flush=True
        )

print("Recorrido terminado.")
print("Créditos disponibles en esta ejecución:", correctos)
print("Consultas con errores:", errores)

Revisadas: 500/681868 (0.1%) | Disponibles: 500 | Errores: 0 | Tiempo: 0.0 min
Revisadas: 1000/681868 (0.1%) | Disponibles: 1000 | Errores: 0 | Tiempo: 0.0 min
Revisadas: 1500/681868 (0.2%) | Disponibles: 1500 | Errores: 0 | Tiempo: 0.0 min
Revisadas: 2000/681868 (0.3%) | Disponibles: 2000 | Errores: 0 | Tiempo: 0.1 min
Revisadas: 2500/681868 (0.4%) | Disponibles: 2500 | Errores: 0 | Tiempo: 0.1 min
Revisadas: 3000/681868 (0.4%) | Disponibles: 3000 | Errores: 0 | Tiempo: 0.1 min
Revisadas: 3500/681868 (0.5%) | Disponibles: 3500 | Errores: 0 | Tiempo: 0.1 min
Revisadas: 4000/681868 (0.6%) | Disponibles: 4000 | Errores: 0 | Tiempo: 0.1 min
Revisadas: 4500/681868 (0.7%) | Disponibles: 4500 | Errores: 0 | Tiempo: 0.1 min
Revisadas: 5000/681868 (0.7%) | Disponibles: 5000 | Errores: 0 | Tiempo: 0.1 min
Revisadas: 5500/681868 (0.8%) | Disponibles: 5500 | Errores: 0 | Tiempo: 0.2 min
Revisadas: 6000/681868 (0.9%) | Disponibles: 6000 | Errores: 0 | Tiempo: 0.2 min
Revisadas: 6500/681868 (1.0%) 

## 7. Resultado de la extracción

- Se descargaron los listados de los 312 meses entre 2000 y 2025,
  con 34.243 páginas y 681.868 películas con identificador único.
- Se obtuvieron los créditos de 681.836 películas. Las 32 restantes
  devolvieron un error 404 y se revisan en el notebook
  `01b_revision_errores`.
- La descarga de créditos, con cuatro trabajadores, tardó 487,8 minutos.